In [1]:
# Install ipython-sql if not already installed
# !pip install ipython-sql

In [2]:
import sqlite3
import sql
import pandas as pd
import numpy as np

# Connect to a database (creates the database file if it doesn't exist)
cnn = sqlite3.connect('northwind1.db')

In [3]:
import sql

# Load the SQL extension
%load_ext sql

# Connect to a SQLite database
%sql sqlite:///northwind1.db

#display style of SQL query results in Jupyter Notebook
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [4]:
%%sql

SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///northwind1.db
Done.


name
sqlite_sequence
CustomerCustomerDemo
CustomerDemographics
EmployeeTerritories
Regions
Territories
Categories
Orders
Products
Shippers


In [5]:
%%sql

SELECT * 
FROM Customers
Limit 5;

 * sqlite:///northwind1.db
Done.


CustomerID,CustomerName,ContactName,Address,City,PostalCode,Country
1,Alfreds Futterkiste,Maria Anders,Obere Str. 57,Berlin,12209,Germany
2,Ana Trujillo Emparedados y helados,Ana Trujillo,Avda. de la Constitución 2222,México D.F.,05021,Mexico
3,Antonio Moreno Taquería,Antonio Moreno,Mataderos 2312,México D.F.,05023,Mexico
4,Around the Horn,Thomas Hardy,120 Hanover Sq.,London,WA1 1DP,UK
5,Berglunds snabbköp,Christina Berglund,Berguvsvägen 8,Luleå,S-958 22,Sweden


In [6]:
%%sql

SELECT * 
FROM Suppliers
Limit 5;

 * sqlite:///northwind1.db
Done.


SupplierID,SupplierName,ContactName,Address,City,PostalCode,Country,Phone
1,Exotic Liquids,Charlotte Cooper,49 Gilbert St.,London,EC1 4SD,UK,(171) 555-2222
2,New Orleans Cajun Delights,Shelley Burke,P.O. Box 78934,New Orleans,70117,USA,(100) 555-4822
3,Grandma Kelly's Homestead,Regina Murphy,707 Oxford Rd.,Ann Arbor,48104,USA,(313) 555-5735
4,Tokyo Traders,Yoshi Nagase,9-8 SekimaiMusashino-shi,Tokyo,100,Japan,(03) 3555-5011
5,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Calle del Rosal 4,Oviedo,33007,Spain,(98) 598 76 54


# Get all suppliers and customers from London. (Using Multiple CTEs)

In [7]:
%%sql

WITH LondonCustomers AS (
    SELECT CustomerID, ContactName, City FROM Customers WHERE City = 'London'
),
LondonSuppliers AS (
    SELECT SupplierID, ContactName, City FROM  WHERE City =Suppliers 'London'
)
SELECT * FROM LondonCustomers
UNION ALL
SELECT * FROM LondonSuppliers;

 * sqlite:///northwind1.db
Done.


CustomerID,ContactName,City
4,Thomas Hardy,London
11,Victoria Ashworth,London
16,Elizabeth Brown,London
19,Ann Devon,London
53,Simon Crowther,London
72,Hari Kumar,London
1,Charlotte Cooper,London


In [8]:
%%sql

WITH LondonCustomers AS (
    SELECT 
        CustomerID AS ID, 
        ContactName, 
        City, 
        'Customer' AS Type
    FROM Customers 
    WHERE City = 'London'
),
LondonSuppliers AS (
    SELECT 
        SupplierID AS ID, 
        ContactName, 
        City, 
        'Supplier' AS Type
    FROM Suppliers 
    WHERE City = 'London'
)
SELECT * FROM LondonCustomers
UNION ALL
SELECT * FROM LondonSuppliers;

 * sqlite:///northwind1.db
Done.


ID,ContactName,City,Type
4,Thomas Hardy,London,Customer
11,Victoria Ashworth,London,Customer
16,Elizabeth Brown,London,Customer
19,Ann Devon,London,Customer
53,Simon Crowther,London,Customer
72,Hari Kumar,London,Customer
1,Charlotte Cooper,London,Supplier


In [9]:
%%sql

SELECT * FROM Products
Limit 10;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,SupplierID,CategoryID,Unit,Price
1,Chai,1,1,10 boxes x 20 bags,18
2,Chang,1,1,24 - 12 oz bottles,19
3,Aniseed Syrup,1,2,12 - 550 ml bottles,10
4,Chef Anton's Cajun Seasoning,2,2,48 - 6 oz jars,22
5,Chef Anton's Gumbo Mix,2,2,36 boxes,21.35
6,Grandma's Boysenberry Spread,3,2,12 - 8 oz jars,25
7,Uncle Bob's Organic Dried Pears,3,7,12 - 1 lb pkgs.,30
8,Northwoods Cranberry Sauce,3,2,12 - 12 oz jars,40
9,Mishi Kobe Niku,4,6,18 - 500 g pkgs.,97
10,Ikura,4,8,12 - 200 ml jars,31


In [11]:
%%sql

SELECT AVG(Price) AS AvgPrice FROM Products;

 * sqlite:///northwind1.db
Done.


AvgPrice
28.866363636363637


In [10]:
%%sql

WITH AvgPriceCTE AS (
    SELECT AVG(Price) AS AvgPrice FROM Products
)
SELECT *
FROM Products
WHERE Price > (SELECT AvgPrice FROM AvgPriceCTE);

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,SupplierID,CategoryID,Unit,Price
7,Uncle Bob's Organic Dried Pears,3,7,12 - 1 lb pkgs.,30
8,Northwoods Cranberry Sauce,3,2,12 - 12 oz jars,40
9,Mishi Kobe Niku,4,6,18 - 500 g pkgs.,97
10,Ikura,4,8,12 - 200 ml jars,31
12,Queso Manchego La Pastora,5,4,10 - 500 g pkgs.,38
17,Alice Mutton,7,6,20 - 1 kg tins,39
18,Carnarvon Tigers,7,8,16 kg pkg.,62.5
20,Sir Rodney's Marmalade,8,3,30 gift boxes,81
26,Gumbär Gummibärchen,11,3,100 - 250 g bags,31.23
27,Schoggi Schokolade,11,3,100 - 100 g pieces,43.9


# 📌 This gives you the number of products in each category.

In [13]:
%%sql

WITH CategoryCounts AS (
    SELECT CategoryID, COUNT(*) AS TotalProducts
    FROM Products
    GROUP BY CategoryID
)
SELECT *
FROM CategoryCounts;

 * sqlite:///northwind1.db
Done.


CategoryID,TotalProducts
1,12
2,12
3,13
4,10
5,7
6,6
7,5
8,12


# CTE to Identify Duplicate Prices
📌 Finds all products that share the same price as others (duplicates).

In [14]:
%%sql

WITH PriceCounts AS (
    SELECT Price, COUNT(*) AS Occurrences
    FROM Products
    GROUP BY Price
    HAVING COUNT(*) > 1
)
SELECT *
FROM Products
WHERE Price IN (SELECT Price FROM PriceCounts) order by price;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,SupplierID,CategoryID,Unit,Price
45,Rogede sild,21,8,1k pkg.,9.5
47,Zaanse koeken,22,3,10 - 4 oz boxes,9.5
3,Aniseed Syrup,1,2,12 - 550 ml bottles,10
21,Sir Rodney's Scones,8,3,24 pkgs. x 4 pieces,10
74,Longlife Tofu,4,7,5 kg pkg.,10
31,Gorgonzola Telino,14,4,12 - 100 g pkgs,12.5
68,Scottish Longbreads,8,3,10 boxes x 8 pieces,12.5
25,NuNuCa Nuß-Nougat-Creme,11,3,20 - 450 g glasses,14
34,Sasquatch Ale,16,1,24 - 12 oz bottles,14
42,Singaporean Hokkien Fried Mee,20,5,32 - 1 kg pkgs.,14


# CTE to Get the Cheapest Product Per Category

In [16]:
%%sql

SELECT CategoryID, MIN(Price) AS MinPrice
    FROM Products
    GROUP BY CategoryID;

 * sqlite:///northwind1.db
Done.


CategoryID,MinPrice
1,4.5
2,10
3,9.2
4,2.5
5,7
6,7.45
7,10
8,6


In [15]:
%%sql

WITH MinPrices AS (
    SELECT CategoryID, MIN(Price) AS MinPrice
    FROM Products
    GROUP BY CategoryID
)
SELECT *
FROM Products
WHERE (CategoryID, Price) IN (
    SELECT CategoryID, MinPrice FROM MinPrices
) order by CategoryID;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,SupplierID,CategoryID,Unit,Price
24,Guaraná Fantástica,10,1,12 - 355 ml cans,4.5
3,Aniseed Syrup,1,2,12 - 550 ml bottles,10
19,Teatime Chocolate Biscuits,8,3,10 boxes x 12 pieces,9.2
33,Geitost,15,4,500 g,2.5
52,Filo Mix,24,5,16 - 2 kg boxes,7
54,Tourtière,25,6,16 pies,7.45
74,Longlife Tofu,4,7,5 kg pkg.,10
13,Konbu,6,8,2 kg box,6


# Get the employees who handle more than 2 territories.

In [17]:
%%sql

WITH TerritoryCount AS (
    SELECT EmployeeID, COUNT(*) AS TotalTerritories
    FROM EmployeeTerritories
    GROUP BY EmployeeID
)
SELECT E.FirstName, E.LastName, T.TotalTerritories
FROM Employees E
JOIN TerritoryCount T ON E.EmployeeID = T.EmployeeID
WHERE T.TotalTerritories > 2;

 * sqlite:///northwind1.db
Done.


FirstName,LastName,TotalTerritories
Andrew,Fuller,7
Janet,Leverling,4
Margaret,Peacock,3
Steven,Buchanan,7
Michael,Suyama,5
Robert,King,10
Laura,Callahan,4
Anne,Dodsworth,7


# 📌 Returns the most expensive product in each category.

In [18]:
%%sql

WITH RankedCategory AS (
    SELECT ProductID, ProductName, CategoryID, Price,
           RANK() OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS RankInCategory
    FROM Products
)
SELECT *
FROM RankedCategory
WHERE RankInCategory = 1;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,RankInCategory
38,Côte de Blaye,1,263.5,1
63,Vegie-spread,2,43.9,1
20,Sir Rodney's Marmalade,3,81,1
59,Raclette Courdavault,4,55,1
56,Gnocchi di nonna Alice,5,38,1
29,Thüringer Rostbratwurst,6,123.79,1
51,Manjimup Dried Apples,7,53,1
18,Carnarvon Tigers,8,62.5,1


# 📌 Filters category 3 products, then returns the top 3 most expensive among them.

In [19]:
%%sql

WITH Filtered AS (
    SELECT * FROM Products WHERE CategoryID = 3
),
Ranked AS (
    SELECT ProductID, ProductName, Price,
           DENSE_RANK() OVER (ORDER BY Price DESC) AS PriceRank
    FROM Filtered
)
SELECT * FROM Ranked WHERE PriceRank <= 3;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,Price,PriceRank
20,Sir Rodney's Marmalade,81,1
62,Tarte au sucre,49.3,2
27,Schoggi Schokolade,43.9,3


In [ ]:
%%sql

SELECT * FROM Products
ORDER BY ProductName;

# 📌 Categorizes products into 'Low', 'Medium', and 'High' price brackets.

In [20]:
%%sql

WITH PriceBuckets AS (
    SELECT ProductID, ProductName, Price,
           CASE 
               WHEN Price < 10 THEN 'Low'
               WHEN Price BETWEEN 10 AND 30 THEN 'Medium'
               ELSE 'High'
           END AS PriceCategory
    FROM Products
)
SELECT *
FROM PriceBuckets
WHERE PriceCategory = 'Low';

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,Price,PriceCategory
13,Konbu,6,Low
19,Teatime Chocolate Biscuits,9.2,Low
23,Tunnbröd,9,Low
24,Guaraná Fantástica,4.5,Low
33,Geitost,2.5,Low
41,Jack's New England Clam Chowder,9.65,Low
45,Rogede sild,9.5,Low
47,Zaanse koeken,9.5,Low
52,Filo Mix,7,Low
54,Tourtière,7.45,Low


# Recursive CTEs

In [21]:
%%sql

WITH RECURSIVE FactorialCTE(n, fact) AS (
    SELECT 1, 1
    UNION ALL
    SELECT n + 1, (n + 1) * fact
    FROM FactorialCTE
    WHERE n < 5
)
SELECT * FROM FactorialCTE;

 * sqlite:///northwind1.db
Done.


n,fact
1,1
2,2
3,6
4,24
5,120


# Return all customers that starts with "E" and are at least 3 characters in length

In [ ]:
%%sql

SELECT * FROM Customers
WHERE CustomerName LIKE 'E__%';

# END..... Practice Below...

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE country = 'Brazil';

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE CustomerID=3;

In [ ]:
%%sql

SELECT * FROM Products;

In [ ]:
%%sql

SELECT *
FROM Products
WHERE Price > 40;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price < 40;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price >= 30;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price <= 30;

In [ ]:
%%sql

SELECT * FROM Products
WHERE Price <> 30;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price BETWEEN 30 AND 40;

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE City IN ('Paris','London');

In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql

